In [6]:
import os
import shutil
from pathlib import Path

REBUILD = False

PROJECT_DIR = Path(r"C:\Users\jakub\Desktop\csgo-rag")
DATA_DIR = PROJECT_DIR / "data"

# jesli poprzednia baza zablokowana, uzyj chroma_db_new
_chroma_new = PROJECT_DIR / "chroma_db_new"
_chroma_main = PROJECT_DIR / "chroma_db"
if _chroma_new.is_dir() and any(_chroma_new.iterdir()):
    CHROMA_DIR = _chroma_new
else:
    CHROMA_DIR = _chroma_main

os.chdir(PROJECT_DIR)
print("Folder projektu:", PROJECT_DIR)
print("Baza wektorowa:", CHROMA_DIR.name)
print("Pliki w data/:", sorted(p.name for p in DATA_DIR.glob("*.md")))

Folder projektu: C:\Users\jakub\Desktop\csgo-rag
Baza wektorowa: chroma_db_new
Pliki w data/: ['bronie.md', 'ekonomia.md', 'mapy.md', 'taktyka.md', 'ustawienia.md', 'utility.md']


In [7]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.vectorstores import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Importy OK")

Importy OK


In [8]:
loader = DirectoryLoader(
    str(DATA_DIR),
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(docs)

print(f"Wczytano {len(docs)} dokumentow, podzielono na {len(chunks)} fragmentow")

Wczytano 6 dokumentow, podzielono na 71 fragmentow


In [9]:
import gc
import time

embeddings = OllamaEmbeddings(model="nomic-embed-text")

if "vectorstore" in globals():
    del vectorstore
gc.collect()
time.sleep(1)

def usun_folder(path):
    if not path.is_dir():
        return True
    for _ in range(5):
        try:
            shutil.rmtree(path)
            return True
        except PermissionError:
            gc.collect()
            time.sleep(1)
    return False

if REBUILD:
    if not usun_folder(CHROMA_DIR):
        CHROMA_DIR = PROJECT_DIR / "chroma_db_new"
        usun_folder(CHROMA_DIR)
        print("Stara baza zablokowana przez Windows. Buduje w chroma_db_new.")
    else:
        print("Wyczyszczono baze:", CHROMA_DIR.name)
    vectorstore = Chroma.from_documents(
        chunks,
        embeddings,
        persist_directory=str(CHROMA_DIR),
    )
    print("Zbudowano nowa baze chroma_db.")
elif CHROMA_DIR.is_dir() and any(CHROMA_DIR.iterdir()):
    vectorstore = Chroma(persist_directory=str(CHROMA_DIR), embedding_function=embeddings)
    print("Zaladowano istniejaca baze:", CHROMA_DIR.name)
else:
    vectorstore = Chroma.from_documents(
        chunks,
        embeddings,
        persist_directory=str(CHROMA_DIR),
    )
    print("Zbudowano nowa baze:", CHROMA_DIR.name)

Zaladowano istniejaca baze: chroma_db_new


C:\Users\jakub\AppData\Local\Temp\ipykernel_25920\157904629.py:37: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=str(CHROMA_DIR), embedding_function=embeddings)


In [10]:
llm = ChatOllama(model="llama3", temperature=0)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


def zapytaj(pytanie):
    dokumenty = retriever.invoke(pytanie)
    kontekst = "\n\n".join(d.page_content for d in dokumenty)
    prompt = (
        "Odpowiedz na pytanie wylacznie na podstawie ponizszego kontekstu. "
        "Jesli czegos nie ma w kontekscie, napisz, ze nie wiesz.\n\n"
        f"KONTEKST:\n{kontekst}\n\nPYTANIE: {pytanie}"
    )
    odpowiedz = llm.invoke(prompt).content
    print("PYTANIE:", pytanie)
    print("\n" + odpowiedz)


print("Asystent gotowy.")

Asystent gotowy.


In [11]:
zapytaj("Ile kosztuje AK-47 i jakie ma obrazenia w glowe?")

PYTANIE: Ile kosztuje AK-47 i jakie ma obrazenia w glowe?

Odpowiem tylko na podstawie kontekstu!

AK-47 kosztuje $2700.

Obrażenia w głowie: zabija jednym strzałem nawet w hełmie (one-tap headshot).


In [12]:
zapytaj("Czym rozni sie M4A4 od M4A1-S?")

PYTANIE: Czym rozni sie M4A4 od M4A1-S?

Na podstawie kontekstu, różnice między M4A4 a M4A1-S to:

* Cena: M4A4 jest droższy o $200 (M4A4 - $3100, M4A1-S - $2900)
* Liczba naboi w magazynku: M4A4 ma 30 nabojów, M4A1-S ma 20 nabojów
* Tłumik: M4A1-S ma tłumik, M4A4 nie ma
* Obrażenia za strzał: M4A1-S ma wyższe obrażenia (38) niż M4A4 (33)
* Charakterystyka: M4A4 jest lepszy w sytuacjach wymagających wielu strzałów na wielu przeciwników, M4A1-S jest lepszy do krótkich wymian ognia i strzałów z ukrycia (tap/burst)


In [13]:
zapytaj("Ile kosztuje AWP i jaka jest premia za zabojstwo z AWP?")

PYTANIE: Ile kosztuje AWP i jaka jest premia za zabojstwo z AWP?

Na podstawie kontekstu, odpowiedź na pytanie brzmi:

Cena AWP: $4750
Premia za zabójstwo z AWP: $100


In [14]:
zapytaj("Co to jest loss bonus i jakie sa jego kwoty?")

PYTANIE: Co to jest loss bonus i jakie sa jego kwoty?

Loss bonus to premia, którą drużyna otrzymuje za przegrane rundy. Kwotami loss bonus są:

* 1. przegrana z rzędu: $1400
* 2. przegrana: $1900
* 3. przegrana: $2400
* 4. przegrana: $2900
* 5. i kolejne: $3400 (maksymalny loss bonus)


In [15]:
zapytaj("Ile kosztuje full buy CT z defuse kitem?")

PYTANIE: Ile kosztuje full buy CT z defuse kitem?

Na podstawie kontekstu, odpowiedź brzmi: ~$5100–$5500.


In [16]:
zapytaj("Jaka jest premia za zabojstwo z nozem i z SMG?")

PYTANIE: Jaka jest premia za zabojstwo z nozem i z SMG?

Na podstawie kontekstu, mogę odpowiedzieć:

Premia za zabójstwo z nożem to $1500 (najwyższa w grze!), a premia za zabójstwo z SMG to $600.


In [17]:
zapytaj("Do czego sluzy molotov i jak dlugo plonie ogien?")

PYTANIE: Do czego sluzy molotov i jak dlugo plonie ogien?

Na podstawie kontekstu, mogę odpowiedzieć na pytanie.

Molotov (Terroryści) służy do blokowania przejścia (np. banana na Inferno, choke point), wykurzania gracza z osłony (force out of cover), opóźniania wejścia wroga na site, przerywania rozbrajania bomby (CT nie może stać w ogniu) i post-plant molotov - ogień na bombę, wymusza wycofanie CT.

Ogień Molotova trwa około 7 sekund.


In [18]:
zapytaj("Co to jest execute i jakie granaty sa uzywane przy ataku na site?")

PYTANIE: Co to jest execute i jakie granaty sa uzywane przy ataku na site?

W kontekście, w którym opisano różne typy granatów i ich zastosowania, "execute" odnosi się do skoordynowanego ataku T-side na bombsite z pełnym wykorzystaniem utility. W tym przypadku, używane są następujące granaty:

1. Smokes - zasłaniają kluczowe pozycje CT (AWP angles, crossfires)
2. Flashes - oślepiają graczy na site przed wejściem
3. Molotov - wykurza CT z osłon (np. molotov car na Dust II B)

Te granaty są używane w celu zasłonięcia pozycji CT, oślepienia ich i wykurzenia z osłon, co umożliwia T-side wykonanie skoordynowanego ataku na bombsite.


In [24]:
zapytaj("Czym zajmuje sie entry fragger i czym jest trade kill?")

PYTANIE: Czym zajmuje sie entry fragger i czym jest trade kill?

W kontekście ekonomii w CS:GO, entry fragger to gracz, który jako pierwszy wchodzi na site, szuka opening kill (pierwszego zabięcia) i dostaje flash od supportu. Trade kill to sytuacja, w której sojusznik ginie, a wchodzisz za niego i zabijasz tego kto go zabił.


In [22]:
zapytaj("Jak ustawic crosshair komendami w konsoli i co to jest eDPI?")

PYTANIE: Jak ustawic crosshair komendami w konsoli i co to jest eDPI?

Ustawić crosshair komendami w konsoli, aby zmienić styl celownika, należy użyć następujących komend:

* `cl_crosshairstyle 4` — statyczny klasyczny (najpopularniejszy u pro graczy)
* `cl_crosshairsize 2` — rozmiar celownika
* `cl_crosshairthickness 1` — grubość linii

Aby wygenerować kod celownika, należy skorzystać z stron (np. crashz crosshair generator) i wkleić go w konsoli.

eDPI to skrót od "effective DPI", czyli efektywna czułość myszy w grze. Jest to iloczyn DPI myszy i sensitivity w grze. Wielu pro graczy gra w przedziale eDPI 600–1000 (np. 400 DPI × 2.0 sens = 800 eDPI).
